In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import joblib
from sklearn.compose import ColumnTransformer

In [2]:
df = pd.read_csv('cleaned_students_dropout01.csv')

In [3]:
df

,School_Type,Location,Infrastructure,Teaching_Staff,Gender,Caste,Age,Standard,Socioeconomic_Status,Dropout_Status
0,Government,Semi-Urban,Poor,Poor,Male,SC,14,8,Low,Dropout
1,Private,Rural,Basic,Good,Female,General,15,7,High,Enrolled
2,Government,Rural,Good,Excellent,Male,ST,13,9,Moderate,Dropout
3,Private,Semi-Urban,Basic,Good,Female,OBC,12,10,High,Enrolled
4,Government,Rural,Basic,Poor,Male,SC,16,8,Low,Dropout
...,...,...,...,...,...,...,...,...,...,...
816,Government,Rural,Basic,Good,Female,ST,15,10,Moderate,Enrolled
817,Government,Rural,Poor,Poor,Male,SC,15,10,High,Enrolled
818,Government,Semi-Urban,Good,Good,Male,General,12,10,High,Enrolled
819,Private,Rural,Basic,Poor,Female,ST,11,9,Low,Dropout


In [4]:
df['Location'].unique()

<StringArray>
['Semi-Urban', 'Rural', 'Urban']
Length: 3, dtype: str

In [5]:
for col in df.columns:
    print(f"{col}: {df[col].unique()} unique values")

School_Type: <StringArray>
['Government', 'Private']
Length: 2, dtype: str unique values
Location: <StringArray>
['Semi-Urban', 'Rural', 'Urban']
Length: 3, dtype: str unique values
Infrastructure: <StringArray>
['Poor', 'Basic', 'Good', 'Excellent']
Length: 4, dtype: str unique values
Teaching_Staff: <StringArray>
['Poor', 'Good', 'Excellent', 'Male', 'Female']
Length: 5, dtype: str unique values
Gender: <StringArray>
['Male', 'Female']
Length: 2, dtype: str unique values
Caste: <StringArray>
['SC', 'General', 'ST', 'OBC']
Length: 4, dtype: str unique values
Age: [14 15 13 12 16 10 11] unique values
Standard: [ 8  7  9 10 12  5  6 11] unique values
Socioeconomic_Status: <StringArray>
['Low', 'High', 'Moderate']
Length: 3, dtype: str unique values
Dropout_Status: <StringArray>
['Dropout', 'Enrolled']
Length: 2, dtype: str unique values


In [6]:
dfCopy = df.copy()


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df.drop('Dropout_Status', axis=1), df['Dropout_Status'], test_size=0.2, random_state=42)

In [8]:
dfCopy.info()

<class 'pandas.DataFrame'>
RangeIndex: 821 entries, 0 to 820
Data columns (total 10 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   School_Type           821 non-null    str  
 1   Location              821 non-null    str  
 2   Infrastructure        821 non-null    str  
 3   Teaching_Staff        821 non-null    str  
 4   Gender                821 non-null    str  
 5   Caste                 821 non-null    str  
 6   Age                   821 non-null    int64
 7   Standard              821 non-null    int64
 8   Socioeconomic_Status  821 non-null    str  
 9   Dropout_Status        821 non-null    str  
dtypes: int64(2), str(8)
memory usage: 64.3 KB


In [9]:
dfCopy.columns

Index(['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender',
       'Caste', 'Age', 'Standard', 'Socioeconomic_Status', 'Dropout_Status'],
      dtype='str')

In [10]:
# cat_col = ['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'gender', 'Caste', 'Socioeconomic_Status']
num_col = ['Age', 'Standard']

In [11]:
Label_cat =['Dropout_Status'] 

In [12]:
dfCopy.columns

Index(['School_Type', 'Location', 'Infrastructure', 'Teaching_Staff', 'Gender',
       'Caste', 'Age', 'Standard', 'Socioeconomic_Status', 'Dropout_Status'],
      dtype='str')

In [13]:
onehot_cols = [
    'School_Type',
    'Location',
    'Gender',
    'Caste'
]
ordinal_cols = [
    'Infrastructure',
    'Teaching_Staff',
    'Socioeconomic_Status'
]

In [14]:
ordinal_categories = [
    ['Poor', 'Basic', 'Good', 'Excellent'],  # Infrastructure
    ['Poor', 'Good', 'Excellent', 'Male', 'Female'],  # Teaching_Staff
    ['Low', 'Moderate', 'High']  # Socioeconomic_Status
]

In [15]:
cat_pipeline = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])
num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

In [16]:
from sklearn.preprocessing import OrdinalEncoder


num_pipeline = Pipeline([
    ('scaler', StandardScaler())
])

onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

ordinal_pipeline = Pipeline([
    ('ordinal', OrdinalEncoder(categories=ordinal_categories))
])
label_pipeline = Pipeline([
    ('label', LabelEncoder())
])


In [17]:
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_col),
    ('onehot', onehot_pipeline, onehot_cols),
    ('ordinal', ordinal_pipeline, ordinal_cols)
])

In [18]:
from sklearn.linear_model import LogisticRegression


model_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

In [19]:
model_pipeline.fit(X_train, y_train)
red = model_pipeline.predict(X_test)
print(classification_report(y_test, red))
print(confusion_matrix(y_test, red))


              precision    recall  f1-score   support

     Dropout       0.87      0.91      0.89        93
    Enrolled       0.88      0.82      0.85        72

    accuracy                           0.87       165
   macro avg       0.87      0.87      0.87       165
weighted avg       0.87      0.87      0.87       165

[[85  8]
 [13 59]]


In [20]:
from sklearn.model_selection import GridSearchCV

rf_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [ 10, 20],
    'model__min_samples_split': [2, 5],
    'model__min_samples_leaf': [1, 2],
    'model__max_features': ['sqrt', 'log2']
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1

)

grid.fit(X_train, y_train)
best_model = grid.best_estimator_

rf_pred = best_model.predict(X_test)
print('Best Params:', grid.best_params_)
print('Best CV Score:', round(grid.best_score_, 4))
print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))

Best Params: {'model__max_depth': 10, 'model__max_features': 'sqrt', 'model__min_samples_leaf': 2, 'model__min_samples_split': 5, 'model__n_estimators': 300}
Best CV Score: 0.939
              precision    recall  f1-score   support

     Dropout       0.94      1.00      0.97        93
    Enrolled       1.00      0.92      0.96        72

    accuracy                           0.96       165
   macro avg       0.97      0.96      0.96       165
weighted avg       0.97      0.96      0.96       165

[[93  0]
 [ 6 66]]


In [21]:
# ! pip install xgboost

In [22]:
from sklearn.model_selection import cross_val_score
cross_val_score(best_model, X_train, y_train, cv=5, scoring='accuracy')

array([0.98484848, 0.95419847, 0.90076336, 0.90839695, 0.94656489])

In [23]:
import joblib
joblib.dump(best_model, "dropout_model.pkl")

['dropout_model.pkl']